# SignalFrame on Colab

Measures a short clip, describes what it measured, and helps you turn that into an experiment
you can run. **It does not predict how an audience will respond**, and no behavioral head is
installed — so no probability appears anywhere in this notebook.

Run the cells in order. The last one prints a public URL you can open from any browser.

---

## Read this before you start: what Colab can and cannot run

Colab is Linux with an NVIDIA GPU. That is genuinely the *better* host for the cortical lane —
the macOS profile is a vision-only ablation, while Colab can run the full path. But two lanes
**cannot run here at all**, and no amount of configuration changes that:

| Lane | On Colab | Why |
| --- | --- | --- |
| Media metadata, measured audio | **Yes** | Needs only `ffmpeg` |
| On-screen text (OCR) | **Yes** | Via the `pytesseract` fallback; Apple Vision is macOS-only |
| Hook readout, recut, variants, experiments | **Yes** | Pure measurement, no model |
| V-JEPA 2.1, AST AudioSet | **Yes, with artifacts** | Pinned checkpoints, hash-verified before load |
| TRIBE v2 cortical | **Yes, with gated access** | Needs accepted terms, a token, and multi-GB weights |
| Insight / Hook Doctor | **Yes, remote only** | `mlx-lm` is Apple-silicon only; use the Anthropic provider |
| Transcript (ASR) | **No** | The adapter is pinned to `mlx-whisper`, Apple-silicon only |
| NanoLLaVA keyframes | **No** | The runtime is `mlx_vlm`, Apple-silicon only |

So this is not "every feature with no limitation" — I would rather say that plainly than have
you discover it three cells in. Everything else works, including the full measure to recut to
re-measure loop.

## 1. Check what hardware you were given

In [ ]:
import subprocess, sys, platform

print("Python :", sys.version.split()[0], "on", platform.platform())
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                          "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
          or "No GPU reported")
except FileNotFoundError:
    print("No GPU on this runtime. Everything except the cortical lane still works;")
    print("set Runtime > Change runtime type > GPU if you intend to install TRIBE v2.")

## 2. Get the code

In [ ]:
import os, subprocess

REPO = "https://github.com/Kayariyan28/Cognitive-Hook-Predictor.git"
BRANCH = "claude/updates-cagd0r"
ROOT = "/content/Cognitive-Hook-Predictor"

if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO, ROOT], check=True)
else:
    subprocess.run(["git", "-C", ROOT, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "pull", "--ff-only"], check=True)

os.chdir(ROOT)
print("HEAD:", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip())

## 3. Install dependencies

The model-free set plus Gradio. `ffmpeg` is what the measured-audio branch needs;
`tesseract-ocr` is what the on-screen-text fallback needs.

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg tesseract-ocr > /dev/null
!pip install -q -r backend/requirements-local.txt
!pip install -q gradio pytesseract pillow

import shutil
for binary in ("ffmpeg", "ffprobe", "tesseract"):
    print(f"{binary:<10}", shutil.which(binary) or "MISSING")

## 4. Configure

Two optional keys. Leave both blank and everything measurement-based still works — the
language lane will simply report itself unavailable, which is the design, not a failure.

- **`ANTHROPIC_API_KEY`** turns on the Hook Doctor. Only derived JSON evidence is sent —
  never your video, audio, frames or tensors. It is off unless you set both switches.
- **`HF_TOKEN`** is only needed for the cortical lane in the optional section at the end.

In [ ]:
import os

ANTHROPIC_API_KEY = ""   # optional: turns on the Hook Doctor
HF_TOKEN = ""            # optional: only for the TRIBE v2 cortical lane

os.environ["INSIGHT_OCR_ENGINE"] = "pytesseract"      # Apple Vision is macOS-only
os.environ["INSIGHT_COMPARATIVE_MINIMUM_CLIPS"] = "5"  # a Colab session has few clips

if ANTHROPIC_API_KEY.strip():
    os.environ["INSIGHT_PROVIDER"] = "anthropic"
    os.environ["INSIGHT_CLOUD_ENABLED"] = "true"
    os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY.strip()
    os.environ["INSIGHT_ANTHROPIC_MODEL"] = "claude-sonnet-4-5-20250929"
    print("Insight lane: remote provider enabled.")
else:
    print("Insight lane: no provider configured. It will report provider_unavailable,")
    print("and every measurement-based feature still works.")

if HF_TOKEN.strip():
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()

## 5. A test clip (optional)

Fourteen seconds, deliberately silent for the first 1.4. That makes the flagged
opening-silence check reproducible, and makes the recut's effect measurable.
Skip this and upload your own clip instead.

In [ ]:
!./scripts/make-demo-clip.sh /content/demo-clip.mp4

## 6. Launch

Starts the API in this process and serves the Gradio interface. `share=True` prints a
public `*.gradio.live` URL — open it in any browser. It stays alive while this cell runs.

In [ ]:
import os, sys
sys.path.insert(0, os.getcwd())

os.environ["SIGNALFRAME_SHARE"] = "true"

from colab.gradio_app import build_interface, start_backend_in_background

start_backend_in_background()
build_interface().launch(share=True, server_name="0.0.0.0", server_port=7860)

---

## What to do in the interface

1. **Analyse** — upload the clip. You will see each branch report available, or unavailable
   with the reason it names. Nothing is substituted for a missing branch.
2. **Hook readout** — the first three seconds as a timeline plus five checks. No model is
   involved. A check that cannot be measured says `not measured`, never `clear`.
3. **Hook Doctor** — cited notes, if you set a key in step 4. Every sentence carries the
   evidence it came from, and the artifact is rejected whole if any citation does not resolve.
4. **Compare cuts** — analyse two cuts, then see which measured signals moved.
5. **Recut** — trim the dead air, download, and analyse the result. The opening-silence check
   should move from flagged to clear, and the silent-window fraction toward zero.

That loop — measure, change one thing, measure again — is the whole product.

---

## Optional: the TRIBE v2 cortical lane

Only worth it if you specifically want the cortical prediction. It is a large, gated install,
and it does **not** improve hook advice: cortical output describes predicted average-subject
BOLD, and this project refuses to treat it as attention, engagement, or performance.

**Before running this**, you must have a Hugging Face account that has accepted the TRIBE v2
terms. The weights are **CC BY-NC 4.0** — non-commercial unless you obtain separate
permission. The checkpoint is hash-verified before load and refused if it does not match.

In [ ]:
# Heavy: torch, the pinned tribev2 package, and multi-GB weights.
# Only run this if you have accepted the gated terms and set HF_TOKEN above.
import os

assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in step 4 first."

!pip install -q torch torchvision
!pip install -q timm einops transformers
!pip install -q -r backend/requirements.txt

# Colab is the deployment this path was designed for: CUDA, and the full
# trimodal mode rather than the macOS vision-only ablation.
os.environ["TRIBE_MODEL_ID"] = "facebook/tribev2"
os.environ["TRIBE_MODEL_REVISION"] = "f894e783020944dcd96e5568550afe2aa9743f9f"
os.environ["TRIBE_CHECKPOINT_NAME"] = "best.ckpt"
os.environ["TRIBE_DEVICE"] = "cuda"
os.environ["TRIBE_VIDEO_DEVICE"] = "cuda"
os.environ["TRIBE_VIDEO_PRECISION"] = "fp32"
os.environ["TRIBE_INFERENCE_MODE"] = "full"

print("Configured. Restart the launch cell for it to take effect.")
print("If the checkpoint digest does not match the audited value, the load is refused")
print("on purpose — that is the integrity check, not a bug.")

---

## Where things are kept

Everything the app writes lives under `backend/.runtime/` in this Colab session and
**disappears when the runtime is recycled**:

- `backend/.runtime/forecast/results/` — published evidence
- `backend/.runtime/insight/artifacts/` — generated insight artifacts and the exact
  evidence each one cites
- `backend/.runtime/insight/rejections/` — refusals, including the offending sentence

Colab is a shared, ephemeral machine. Treat anything you upload as leaving your control:
for client footage or anything sensitive, run it locally with `./scripts/start-mac.sh`
instead, where nothing leaves the machine unless you switch on the remote provider.